# 5 — Diffusion par Racine Carrée (CIR)

## Contexte

Le modèle de **Cox-Ingersoll-Ross (1985)** est utilisé pour modéliser des taux courts
stochastiques avec retour à la moyenne. Il étend le GBM en rendant la volatilité
proportionnelle à $\sqrt{x_t}$, ce qui garantit que le processus reste **positif**.

---

## Intuition Financière — Pourquoi CIR pour les taux ?

Les taux d'intérêt ne se comportent pas comme des prix d'actions :

- Ils ne peuvent pas être **négatifs** (sous le modèle CIR standard)
- Ils ne **divergent pas** indéfiniment — un taux à 50% finit toujours par redescendre
- Ils gravitent autour d'un **niveau d'équilibre** économique de long terme $\theta$

> Le GBM est inadapté pour les taux : il permet des valeurs négatives et n'a pas de
> force de rappel. CIR résout ces deux problèmes simultanément.
> Le modèle CIR est aussi utilisé comme processus de variance dans le modèle de **Heston**
> (volatilité stochastique), ce qui en fait un outil central en finance de marché.

### Analogie : le ressort (loi de Hooke)

Le drift $\kappa(\theta - x_t)$ se comporte comme un **ressort** :

| Situation | Force de rappel | Direction |
|---|---|---|
| $x_t \gg \theta$ | $(\theta - x_t) < 0$ | ↓ vers le bas |
| $x_t = \theta$ | $(\theta - x_t) = 0$ | → pas de force |
| $x_t \ll \theta$ | $(\theta - x_t) > 0$ | ↑ vers le haut |

Plus $\kappa$ est grand, plus le ressort est rigide (retour rapide vers $\theta$).

### Pourquoi √x_t dans la volatilité ?

| Niveau du taux | Volatilité effective $\sigma\sqrt{x_t}$ | Interprétation |
|---|---|---|
| $x_t$ élevé | Grande | Volatilité plus forte quand les taux sont hauts |
| $x_t \to 0$ | $\to 0$ | La volatilité s'annule → barrière naturelle en 0 |
| $x_t < 0$ | Impossible | Le modèle ne peut pas franchir zéro |

---

## Équation Différentielle Stochastique (Eq. 18-5)

$$dx_t = \kappa(\theta - x_t)\,dt + \sigma\sqrt{x_t}\,dZ_t$$

### Décomposition des termes

| Terme | Nom | Rôle |
|---|---|---|
| $\kappa(\theta - x_t)\,dt$ | **Drift à retour à la moyenne** | Force de rappel vers $\theta$. Plus $x_t$ s'éloigne de $\theta$, plus le rappel est fort. $\kappa$ contrôle la vitesse. |
| $\sigma\sqrt{x_t}\,dZ_t$ | **Diffusion** | Volatilité proportionnelle à $\sqrt{x_t}$. S'annule quand $x_t \to 0$, ce qui empêche le processus de devenir négatif. |

### Paramètres

| Symbole | Signification |
|---|---|
| $x_t$ | Valeur du processus au temps $t$ (ex : taux court) |
| $\kappa$ | Vitesse de retour à la moyenne (*mean-reversion speed*) |
| $\theta$ | Niveau moyen de long terme (*long-term mean*) |
| $\sigma$ | Paramètre de volatilité |
| $dZ_t$ | Incrément du mouvement brownien standard |

### Condition de Feller

$$2\kappa\theta \geq \sigma^2$$

> Si cette condition est respectée, $x_t > 0$ **strictement** pour tout $t$.
> Si elle est violée, le processus peut atteindre zéro — d'où l'importance
> de la troncature dans la discrétisation numérique.

---

## Schéma de Discrétisation — Full Truncation Scheme (Eq. 18-6)

$$\tilde{x}_{t_{m+1}} = \tilde{x}_{t_m} + \kappa(\theta - \tilde{x}^+_{t_m})(t_{m+1} - t_m) + \sigma\sqrt{\tilde{x}^+_{t_m}}\sqrt{t_{m+1} - t_m}\, z_t$$

$$x_{t_{m+1}} = \tilde{x}^+_{t_{m+1}}$$

### Logique du schéma pas à pas

| Étape | Opération | Rôle |
|---|---|---|
| **1** | $\tilde{x}^+_{t_m} = \max(\tilde{x}_{t_m},\ 0)$ | Troncature **avant** le calcul — protège le drift ET la diffusion |
| **2** | Calcul de $\tilde{x}_{t_{m+1}}$ via l'équation d'Euler | Pas de simulation |
| **3** | $x_{t_{m+1}} = \max(\tilde{x}_{t_{m+1}},\ 0)$ | Troncature **après** le calcul — valeur stockée toujours $\geq 0$ |

### Points clés

- $\tilde{x}^+ = \max(\tilde{x},\, 0)$ : troncature appliquée **deux fois** — avant (dans le calcul) et après (sur le résultat)
- $z_t \sim \mathcal{N}(0, 1)$ : tirage aléatoire standard à chaque pas
- Le terme $\sqrt{t_{m+1} - t_m}$ est le **scaling brownien** : la volatilité croît en $\sqrt{\Delta t}$

---

## Comparaison des Modèles de Diffusion

| Modèle | SDE | Volatilité | Valeurs négatives ? | Usage typique |
|---|---|---|---|---|
| GBM | $dS = \mu S\,dt + \sigma S\,dZ$ | $\sigma S_t$ | Non (si $S_0 > 0$) | Prix d'actions |
| Jump Diffusion | $dS = (r - r_J)S\,dt + \sigma S\,dZ + JS\,dN$ | $\sigma S_t$ + sauts | Non | Actions avec chocs |
| Vasicek | $dr = \kappa(\theta-r)\,dt + \sigma\,dZ$ | $\sigma$ (constante) | **Oui** | Taux (obsolète) |
| **CIR** | $dx = \kappa(\theta-x)\,dt + \sigma\sqrt{x}\,dZ$ | $\sigma\sqrt{x_t}$ | Non (si Feller OK) | **Taux courts, variance Heston** |

> Le modèle de **Vasicek** (1977) est le précurseur direct du CIR — même drift,
> mais volatilité constante qui autorise les taux négatifs. CIR corrige ce défaut.

